# 05 — Silver Orders with Data Quality + Quarantine

## Overview
This lab adds a **rule-based data quality gate** between Bronze and Silver. Rather than letting bad data silently corrupt the Silver table, it routes failing rows to a quarantine zone for investigation while allowing clean rows to continue to the MERGE step.

**What you will learn**
- How to define a declarative DQ rule set with three enforcement modes: `fail_fast` (pipeline abort), `warn` (tag & quarantine), and `continue` (tag & quarantine, softer)
- How to evaluate rules in two passes: abort-early on critical failures, then tag soft failures per row
- How to write quarantined rows to both a Delta table (queryable) and raw Parquet files (re-injectable)
- How `_dq_failures` — an array column listing every failed rule ID — enables efficient root-cause queries
- How to integrate quality checks into the same Silver MERGE used in Scenario 02 without duplicating logic

**Prerequisites:** Scenario 01 has been run; `bronze.orders_raw` is populated. First run will also create `bronze.orders_quarantine` and `bronze.pipeline_errors`.

### Cell 1 — Ingest-date parameter
Scopes quality checks to a specific Bronze partition. When empty, the full Bronze table is evaluated — useful for a retroactive quality sweep or when running the lab for the first time. In production, the Data Factory pipeline injects today's date so only the latest batch is checked, keeping run times bounded.

In [ ]:
p_ingest_date = ""   # empty = all unprocessed partitions

### Cell 2 — Configuration & table paths
Sets up five table references. `QUARANTINE_PATH` is a raw file path (separate from the Delta table) so that remediation teams can download quarantined files directly from OneLake and re-submit them after correction — without needing Spark access.

In [ ]:
spark.conf.set("spark.sql.shuffle.partitions", "8")
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import date

LH              = "lh_advanced_scenarios"
BRONZE          = f"{LH}.bronze.orders_raw"
SILVER          = f"{LH}.silver.orders"
QUARANTINE_TBL  = f"{LH}.bronze.orders_quarantine"
QUARANTINE_PATH = "Files/quarantine/orders"
TODAY           = date.today().isoformat()

### Cell 3 — DQ rule definitions
Declares the quality rule set as a list of `(rule_id, PySpark column expression, fail_mode)` tuples. Three modes control pipeline behaviour:
- **`fail_fast`** — any violation immediately aborts the pipeline and logs to `pipeline_errors`. Appropriate for business keys (`order_id`, `customer_id`) where a missing value makes the row unprocessable.
- **`warn`** — violations are tagged with the rule ID and routed to quarantine, but the pipeline continues. Appropriate for soft constraints like `amount ≥ 0` or valid `status` values, where the order is still meaningful even with a bad field.
- **`continue`** — same as `warn` but signals that the issue is expected to resolve over time (e.g. orders with dates slightly in the future due to timezone drift).

In [ ]:
# ── DQ Rule definitions ───────────────────────────────────────────────────────
# Each rule: (rule_id, column_expr, fail_mode)
# fail_mode: 'fail_fast' | 'warn' | 'continue'
VALID_STATUSES = ["NEW", "SHIPPED", "CANCELLED", "RETURNED"]

DQ_RULES = [
    ("R01", F.col("order_id").isNotNull(),                          "fail_fast"),
    ("R02", F.col("updated_at").isNotNull(),                        "fail_fast"),
    ("R06", F.col("customer_id").isNotNull(),                       "fail_fast"),
    ("R03", F.col("amount").isNull() | (F.col("amount") >= 0),      "warn"),
    ("R04", F.upper(F.trim(F.col("status"))).isin(VALID_STATUSES),  "warn"),
    ("R05", F.col("order_date").between("2000-01-01", TODAY),       "continue"),
]

### Cell 4 — Read Bronze
Loads the Bronze rows to evaluate. Printing the count early is the first sanity check — if it shows far more rows than expected, a pipeline parameter may be wrong (e.g. missing date filter). If zero, the source file was not landed yet. Both conditions are worth catching before spending compute on rule evaluation.

In [ ]:
# ── Read Bronze ───────────────────────────────────────────────────────────────
if p_ingest_date:
    df = spark.read.format("delta").table(BRONZE).filter(F.col("_ingest_date") == p_ingest_date)
else:
    df = spark.read.format("delta").table(BRONZE)

print(f"Input rows: {df.count():,}")

In [ ]:
# ── Apply DQ rules ────────────────────────────────────────────────────────────
df_checked = df

# Check fail_fast rules first — abort immediately on any violation
for rule_id, condition, mode in DQ_RULES:
    if mode == "fail_fast":
        violations = df_checked.filter(~condition).count()
        if violations > 0:
            msg = f"DQ FAIL_FAST [{rule_id}]: {violations} violations found."
            print(f"ERROR: {msg}")
            spark.createDataFrame([{
                "pipeline_name": "05_silver_orders_quality",
                "entity_name":   "orders",
                "error_message": msg,
                "logged_at":     F.current_timestamp(),
            }]).write.format("delta").mode("append").saveAsTable(f"{LH}.bronze.pipeline_errors")
            raise Exception(msg)

# Add per-row failure tags for warn/continue rules
df_tagged = df_checked
failure_flags = []
for rule_id, condition, mode in DQ_RULES:
    if mode in ("warn", "continue"):
        flag_col = f"_fail_{rule_id}"
        df_tagged = df_tagged.withColumn(flag_col, ~condition)
        failure_flags.append((flag_col, rule_id))

# Build _dq_failures array
failure_exprs = [
    F.when(F.col(col), F.lit(rule_id)) for col, rule_id in failure_flags
]
df_tagged = df_tagged.withColumn(
    "_dq_failures",
    F.array_compact(F.array(*failure_exprs))
).withColumn(
    "_dq_passed",
    F.size(F.col("_dq_failures")) == 0
)

passed_count   = df_tagged.filter(F.col("_dq_passed") == True).count()
rejected_count = df_tagged.filter(F.col("_dq_passed") == False).count()
print(f"Passed: {passed_count:,} | Rejected: {rejected_count:,}")

In [ ]:
# ── Route FAILED rows to quarantine ─────────────────────────────────────────
df_rejected = (
    df_tagged.filter(F.col("_dq_passed") == False)
             .drop(*[col for col, _ in failure_flags], "_dq_passed")
             .withColumn("_quarantine_reason", F.array_join(F.col("_dq_failures"), ","))
             .withColumn("_quarantine_ts",     F.current_timestamp())
             .drop("_dq_failures")
)

if rejected_count > 0:
    # Write to Delta quarantine table
    df_rejected.write.format("delta").mode("append").saveAsTable(QUARANTINE_TBL)
    # Write raw files for downstream remediation
    df_rejected.write.format("parquet").mode("append") \
               .save(f"{QUARANTINE_PATH}/{TODAY}/")
    print(f"Quarantined {rejected_count:,} rows → {QUARANTINE_TBL}")

In [ ]:
# ── Route PASSED rows → Silver (same MERGE as Scenario 02) ──────────────────
df_passed = (
    df_tagged
    .filter(F.col("_dq_passed") == True)
    .drop(*[col for col, _ in failure_flags], "_dq_passed", "_dq_failures")
    .withColumn("status",           F.upper(F.trim(F.col("status"))))
    .withColumn("customer_id_hash", F.sha2(F.col("customer_id"), 256))
    .drop("customer_id", "_ingest_date", "_ingest_ts", "_source_file")
    .withColumn("amount",    F.coalesce(F.col("amount"), F.lit(0.00)))
    .withColumn("_silver_ts", F.current_timestamp())
)

# Dedup
w = Window.partitionBy("order_id").orderBy(F.col("updated_at").desc())
df_deduped = df_passed.withColumn("_rn", F.row_number().over(w)) \
                      .filter(F.col("_rn") == 1).drop("_rn")

if not spark.catalog.tableExists(SILVER):
    df_deduped.write.format("delta").mode("overwrite") \
              .option("overwriteSchema", "true").saveAsTable(SILVER)
else:
    silver_dt = DeltaTable.forName(spark, SILVER)
    (
        silver_dt.alias("t")
                 .merge(df_deduped.alias("s"), "t.order_id = s.order_id")
                 .whenMatchedUpdate(condition="s.updated_at > t.updated_at", set={
                     c: f"s.{c}" for c in ["order_date","updated_at","status","amount","customer_id_hash","_silver_ts"]
                 })
                 .whenNotMatchedInsertAll()
                 .execute()
    )

print(f"Silver MERGE complete. Passed rows processed: {passed_count:,}")